# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

✅ EmbeddingManager initialized
🔐 API Key: Loaded from secrets
🌐 Base URL: https://api.openai.com/v1
🤖 Model: gpt-4o
Loading embeddings
embeddings loaded.


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[LAB] Water Generation - Tank QAS and Params", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: [LAB] Water Generation - Tank  QAS and Params, Similarity: 0.90, Type: Diagram, Phase: Logical Architecture LA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
#import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## ⚙️ Execute  Prompt


Execute the prompt that will use the model and system document. 

In [5]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)#analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
What would be the impact based on the model if the Recieve Water Value on water consumed went for 75 to 200 liters of water. 
-What consideration should made specifically to the storage tank.
-What analysis should be performed.
-What model elements with be effected.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

✅ ChatGPTAnalyzer initialized
🔐 API Key: Loaded from secrets
🌐 Base URL: https://api.openai.com/v1
🤖 Model: gpt-4o


**Your prompt:** 
What would be the impact based on the model if the Recieve Water Value on water consumed went for 75 to 200 liters of water. 
-What consideration should made specifically to the storage tank.
-What analysis should be performed.
-What model elements with be effected.


**Response:**

To assess the impact of increasing the "Receive Water" value from 75 to 200 liters of water, several considerations and analyses need to be made, particularly focusing on the storage tank and related model elements. Here's a detailed breakdown:

### Considerations for the Storage Tank:
1. **Capacity Assessment:**
   - Ensure that the storage tank has sufficient capacity to handle the increased water volume. The current requirement states that the storage container should be capable of containing at least X liters of water. This value needs to be updated to reflect the new requirement of 200 liters.

2. **Structural Integrity:**
   - Evaluate the structural integrity of the storage tank to ensure it can withstand the increased weight and pressure from the additional water volume.

3. **Material Durability:**
   - Assess the materials used in the construction of the storage tank to ensure they are suitable for the increased load and potential wear over time.

4. **Design Modifications:**
   - Consider any necessary design modifications to accommodate the increased capacity, such as reinforcing the tank or altering its dimensions.

### Analysis to be Performed:
1. **Capacity Analysis:**
   - Perform a detailed capacity analysis to determine if the current storage tank design can accommodate 200 liters. This includes volume calculations and stress testing.

2. **Flow Rate Analysis:**
   - Analyze the flow rate of water into and out of the storage tank to ensure that the system can handle the increased volume without bottlenecks or overflow issues.

3. **Impact on Functional Chains:**
   - Evaluate how the increased water volume affects the functional chains, particularly the "Water Management" chain, to ensure that all components can handle the increased demand.

4. **Risk Assessment:**
   - Conduct a risk assessment to identify potential failure points or risks associated with the increased water volume, such as leaks or structural failure.

### Model Elements Affected:
1. **Storage Container:**
   - The "Storage Container" logical component will be directly affected, requiring updates to its capacity and possibly its design.

2. **Store the Water Function:**
   - The "Store the Water" logical function will need to be reviewed to ensure it can handle the increased input and output of water.

3. **Water Consumer Property Value Group:**
   - The "Water Consumer" property value group, which includes the consumption rate, will need to be updated to reflect the new water consumption values.

4. **Functional Exchanges:**
   - Any functional exchanges involving water, such as those between "Store the Water" and "Receive Water" functions, will need to be reviewed to ensure they can handle the increased volume.

5. **Requirements:**
   - The "Size" requirement for the storage container will need to be updated to specify the new capacity requirement of at least 200 liters.

6. **Component Exchanges:**
   - Component exchanges related to water flow, such as "Water Pour" and "Water Pouring," may need to be adjusted to accommodate the increased volume.

By addressing these considerations and analyses, the system can be effectively adapted to handle the increased water volume, ensuring continued functionality and reliability.

**Token Usage Info:**

Tokens used: prompt=48513, completion=633, total=49146

## 💬 Launch Interactive Chat on Structured Input




In [6]:
print("Done")

Done


# 